In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../Data/Train.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1000, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,C,Flight,2,5,306,6,high,M,45,3838,1
1,2,F,Ship,7,2,114,3,high,M,35,2710,1
2,3,B,Ship,7,2,215,7,medium,F,44,4152,0
3,4,A,Road,5,3,126,5,medium,M,54,2245,0
4,5,F,Ship,3,5,113,3,medium,M,43,1806,1


In [3]:
X = df.drop("Reached.on.Time_Y.N", axis=1)
y = df["Reached.on.Time_Y.N"]
X = pd.get_dummies(X, drop_first=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

In [6]:
select_pipeline = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=2))
])

In [7]:
combined_features = FeatureUnion([
    ("pca_features", pca_pipeline),
    ("selected_features", select_pipeline)
])

In [8]:
X_combined = combined_features.fit_transform(X_train, y_train)

print("Combined Feature Shape:", X_combined.shape)

Combined Feature Shape: (800, 4)


In [9]:
model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

In [10]:
model.fit(X_combined, y_train)

print("Training Score:", model.score(X_combined, y_train))

predictions = model.predict(X_combined)

print("First 5 Predictions:", predictions[:5])

Training Score: -0.018571223527172442
First 5 Predictions: [0.40181868 0.47009087 0.35225366 0.393479   0.42078932]
